# HGB A/B Ensemble with Isotonic Calibration
제출형 스크립트를 노트북으로 변환했습니다. 각 셀은 원본 코드 구조를 그대로 보존합니다.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, sys, time, json, warnings
warnings.filterwarnings("ignore")

from typing import Tuple, List, Sequence
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn import __version__ as sklver

# -------------------
# 고정 경로
# -------------------
DATA_DIR = "data"
OUTPUT_DIR = "output"
MODEL_DIR = "model"
SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.csv")
A_MODEL_PATH = os.path.join(MODEL_DIR, "model_A.joblib")
B_MODEL_PATH = os.path.join(MODEL_DIR, "model_B.joblib")
A_PREPROC_PATH = os.path.join(MODEL_DIR, "preproc_A.joblib")
B_PREPROC_PATH = os.path.join(MODEL_DIR, "preproc_B.joblib")
META_PATH = os.path.join(MODEL_DIR, "meta.json")

RANDOM_STATE = 42

# -------------------
# 실행 옵션
# -------------------
USE_CALIBRATION = True          
CALIB_METHOD = "isotonic"      
CALIB_CV = 3                    
TEST_SIZE_FOR_LOG = 0.1         

ENSEMBLE_SEEDS: Sequence[int] = (42, 202, 777)

BASE_HGB_PARAMS = dict(
    learning_rate=0.06,          
    max_iter=300,               
    max_depth=None,
    max_leaf_nodes=63,           
    min_samples_leaf=20,        
    l2_regularization=0.0,
    early_stopping=True,
    validation_fraction=0.12,
    n_iter_no_change=25,
    class_weight="balanced",    
)

# -------------------
# 보조 유틸
# -------------------
def ensure_dirs():
    os.makedirs(MODEL_DIR, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

def read_index_files() -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_idx = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
    test_idx  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
    return train_idx, test_idx

def read_feature_files(split: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    A_df = pd.read_csv(os.path.join(DATA_DIR, split, "A.csv"))
    B_df = pd.read_csv(os.path.join(DATA_DIR, split, "B.csv"))
    return A_df, B_df

def separate_num_cat(df: pd.DataFrame, drop_cols: List[str]) -> Tuple[List[str], List[str]]:
    cols = [c for c in df.columns if c not in drop_cols]
    cat_cols = [c for c in cols if str(df[c].dtype) in ("object", "category")]
    num_cols = [c for c in cols if c not in cat_cols]
    return num_cols, cat_cols

def build_preprocessor(num_cols: List[str], cat_cols: List[str]) -> ColumnTransformer:
    numeric_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])
    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordenc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])
    preproc = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, num_cols),
            ("cat", categorical_pipe, cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )
    return preproc

def _mk_calibrator(base_clf):
    # sklearn 1.4+ : estimator, 1.3- : base_estimator
    try:
        major, minor, *_ = map(int, sklver.split(".")[:2])
    except Exception:
        major, minor = 1, 4
    kw = dict(method=CALIB_METHOD, cv=CALIB_CV)
    if (major, minor) >= (1, 4):
        return CalibratedClassifierCV(estimator=base_clf, **kw)
    else:
        return CalibratedClassifierCV(base_estimator=base_clf, **kw)

def maybe_calibrate(base_clf, X_train, y_train):
    if not USE_CALIBRATION:
        return base_clf
    calib = _mk_calibrator(base_clf)
    calib.fit(X_train, y_train)
    return calib

def add_rowwise_features(df: pd.DataFrame, feature_cols: List[str]) -> pd.DataFrame:
    X = df[feature_cols]
    na_count = X.isna().sum(axis=1).astype(np.int32)
    na_ratio = (na_count / (len(feature_cols) + 1e-9)).astype(np.float32)
    df2 = df.copy()
    df2["NA_COUNT"] = na_count
    df2["NA_RATIO"] = na_ratio
    return df2

def build_model(seed: int) -> HistGradientBoostingClassifier:
    params = BASE_HGB_PARAMS.copy()
    params["random_state"] = seed
    return HistGradientBoostingClassifier(**params)

class AvgProbaEnsemble:
    def __init__(self, models: List):
        self.models = models

    def predict_proba(self, X):
        probs = [m.predict_proba(X) for m in self.models]
        return np.mean(probs, axis=0)

# -------------------
# 학습/로드
# -------------------
def fit_or_load(
    df_feat: pd.DataFrame,
    df_idx: pd.DataFrame,
    label_col: str,
    model_path: str,
    preproc_path: str,
    which: str
):
    key = "Test_id"
    assert key in df_feat.columns, f"{which}: '{key}' not found in features"

    if len(df_idx) and label_col in df_idx.columns:
        df = df_idx.merge(df_feat, on=key, how="left", validate="1:1")
        drop_cols = [key, label_col] + (["Test"] if "Test" in df.columns else [])

        feature_cols = [c for c in df.columns if c not in drop_cols]
        df = add_rowwise_features(df, feature_cols)

        num_cols, cat_cols = separate_num_cat(df, drop_cols)
        preproc = build_preprocessor(num_cols, cat_cols)

        X = df.drop(columns=drop_cols)
        y = df[label_col].astype(int).values

        X_tr, X_val, y_tr, y_val = train_test_split(
            X, y, test_size=TEST_SIZE_FOR_LOG, random_state=RANDOM_STATE, stratify=y
        )

        X_tr_t = preproc.fit_transform(X_tr)
        X_val_t = preproc.transform(X_val)

        # 앙상블 학습
        members = []
        for sd in ENSEMBLE_SEEDS:
            base = build_model(sd).fit(X_tr_t, y_tr)
            mdl = maybe_calibrate(base, X_tr_t, y_tr)
            members.append(mdl)
        ensemble = AvgProbaEnsemble(members)

        try:
            val_proba = np.clip(ensemble.predict_proba(X_val_t)[:, 1], 1e-7, 1-1e-7)
            auc = roc_auc_score(y_val, val_proba)
            brier = brier_score_loss(y_val, val_proba)
            print(f"[{which}] Holdout AUC={auc:.5f}, Brier={brier:.5f}")
        except Exception as e:
            print(f"[{which}] validation logging skipped: {e}")

        joblib.dump(preproc, preproc_path)
        joblib.dump(ensemble, model_path)
        print(f"[{which}] trained and saved → {model_path}, {preproc_path}")
        return preproc, ensemble

    print(f"[{which}] loading pre-trained: {preproc_path}, {model_path}")
    preproc = joblib.load(preproc_path)
    ensemble = joblib.load(model_path)
    return preproc, ensemble

# -------------------
# 추론
# -------------------
def predict_partition(
    df_feat: pd.DataFrame,
    df_idx: pd.DataFrame,
    preproc,
    clf_or_ens, 
    which: str
) -> pd.DataFrame:
    key = "Test_id"
    df = df_idx.merge(df_feat, on=key, how="left", validate="1:1")
    drop_cols = [key] + (["Test"] if "Test" in df.columns else [])

    feature_cols = [c for c in df.columns if c not in drop_cols]
    df = add_rowwise_features(df, feature_cols)

    X = df.drop(columns=drop_cols, errors="ignore")
    X_t = preproc.transform(X)
    proba = np.clip(clf_or_ens.predict_proba(X_t)[:, 1], 1e-7, 1-1e-7)
    out = df_idx[[key]].copy()
    out["Label"] = proba
    out["__which__"] = which
    return out

def save_meta():
    meta = dict(
        model="HGB(3-seed soft ensemble) + OrdinalEnc + Calibration",
        hgb_params=BASE_HGB_PARAMS,
        ensemble_seeds=list(ENSEMBLE_SEEDS),
        use_calibration=USE_CALIBRATION,
        calib_method=CALIB_METHOD,
        calib_cv=CALIB_CV,
        sklearn_version=sklver,
        random_state=RANDOM_STATE,
    )
    with open(META_PATH, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

# -------------------
# 메인
# -------------------
def main():
    t0 = time.time()
    ensure_dirs()

    train_idx, test_idx = read_index_files()
    A_train_feat, B_train_feat = read_feature_files("train")
    A_test_feat,  B_test_feat  = read_feature_files("test")

    need_train_A = not (os.path.exists(A_MODEL_PATH) and os.path.exists(A_PREPROC_PATH))
    need_train_B = not (os.path.exists(B_MODEL_PATH) and os.path.exists(B_PREPROC_PATH))

    # A
    A_train_idx = train_idx[train_idx["Test"] == "A"].copy()
    A_test_idx  = test_idx[test_idx["Test"] == "A"].copy()
    if need_train_A:
        print("[A] training path (no pre-trained weights found).")
        fit_or_load(A_train_feat, A_train_idx, "Label", A_MODEL_PATH, A_PREPROC_PATH, "A")
    preproc_A, clf_A = fit_or_load(A_train_feat, pd.DataFrame({"Test_id": []}), "Label",
                                   A_MODEL_PATH, A_PREPROC_PATH, "A")

    # B
    B_train_idx = train_idx[train_idx["Test"] == "B"].copy()
    B_test_idx  = test_idx[test_idx["Test"] == "B"].copy()
    if need_train_B:
        print("[B] training path (no pre-trained weights found).")
        fit_or_load(B_train_feat, B_train_idx, "Label", B_MODEL_PATH, B_PREPROC_PATH, "B")
    preproc_B, clf_B = fit_or_load(B_train_feat, pd.DataFrame({"Test_id": []}), "Label",
                                   B_MODEL_PATH, B_PREPROC_PATH, "B")

    preds_A = predict_partition(A_test_feat, A_test_idx, preproc_A, clf_A, "A") if len(A_test_idx) else None
    preds_B = predict_partition(B_test_feat, B_test_idx, preproc_B, clf_B, "B") if len(B_test_idx) else None

    if preds_A is not None and preds_B is not None:
        sub = pd.concat([preds_A, preds_B], axis=0, ignore_index=True)
    elif preds_A is not None:
        sub = preds_A.copy()
    elif preds_B is not None:
        sub = preds_B.copy()
    else:
        sub = test_idx[["Test_id"]].copy()
        sub["Label"] = 0.001

    try:
        sample = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
        sub = sub.merge(sample[["Test_id"]], on="Test_id", how="right")
        sub = sub[["Test_id", "Label"]]
    except Exception:
        sub = sub[["Test_id", "Label"]]

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    sub.to_csv(SUBMISSION_PATH, index=False)
    save_meta()

    dt = time.time() - t0
    print(f"[이이이잉] submission saved -> {SUBMISSION_PATH} | elapsed: {dt:.2f}s")

if __name__ == "__main__":
    main()
